# 年报下载与NLP提取计算情绪指标
每次手动运行一批（100家公司），运行前修改 `BATCH_START` 和 `BATCH_END`

---
## 安装依赖（首次运行执行一次）

In [ ]:
!pip install pdfplumber transformers torch sentencepiece -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 67.7 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).



###Cell 1：筛选医药生物行业，生成 master2

In [ ]:
import pandas as pd
import os

# 路径配置
BASE        = '/content/drive/MyDrive/credit_risk_project'
MASTER_PATH = os.path.join(BASE, 'finance_data/master_table.csv')
MASTER2_PATH= os.path.join(BASE, 'master2.csv')
LOG_DIR     = os.path.join(BASE, 'logs')
PDF_DIR     = os.path.join(BASE, 'annual_report_pdf')

# 建立必要文件夹
os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(PDF_DIR, exist_ok=True)

# 目标行业
TARGET_INDUSTRIES = ['医药生物']

# 读取并筛选
master = pd.read_csv(MASTER_PATH, dtype={'股票代码': str})
master2 = master[master['申万一级行业'].isin(TARGET_INDUSTRIES)].copy()

# 删除当前ST企业
st_codes = master2[
    master2['股票简称'].str.contains(r'^\*?ST', regex=True, na=False)
]['股票代码'].unique()
master2 = master2[~master2['股票代码'].isin(st_codes)].copy()

# 添加自然编号（按股票代码去重后排序）
unique_stocks = (
    master2[['股票代码', '股票简称', '申万一级行业']]
    .drop_duplicates(subset='股票代码')
    .sort_values('股票代码')
    .reset_index(drop=True)
)
unique_stocks['自然编号'] = unique_stocks.index + 1

# 加入NLP结果占位列（首次建立）
if not os.path.exists(MASTER2_PATH):
    master2 = master2.merge(
        unique_stocks[['股票代码', '自然编号']], on='股票代码', how='left'
    )
    # NLP特征占位列
    for col in ['sentiment_score', 'positive_ratio', 'negative_ratio',
                'risk_word_ratio', 'uncertainty_ratio', 'mda_length', 'nlp_status']:
        master2[col] = None
    master2.to_csv(MASTER2_PATH, index=False, encoding='utf-8-sig')
    print(f'master2 已建立，保存至 {MASTER2_PATH}')
else:
    master2 = pd.read_csv(MASTER2_PATH, dtype={'股票代码': str})
    print(f'master2 已存在，直接读取')

# 统计
print(f'\n总行数：{len(master2)}')
print(f'总公司数：{master2["股票代码"].nunique()}')
print(f'\n各行业公司数：')
print(master2.drop_duplicates("股票代码")["申万一级行业"].value_counts())
print(f'\n前5行预览：')
master2.head()

master2 已建立，保存至 /content/drive/MyDrive/credit_risk_project/master2.csv

总行数：2746
总公司数：458

各行业公司数：
申万一级行业
医药生物    458
Name: count, dtype: int64

前5行预览：


,股票代码,股票简称,报告期,申万一级行业,营业总收入,营业利润,利润总额,净利润,资产-货币资金,资产-总资产,...,净现金流-净现金流,经营性现金流-现金流量净额,自然编号,sentiment_score,positive_ratio,negative_ratio,risk_word_ratio,uncertainty_ratio,mda_length,nlp_status
0,000028,国药一致,2020-12-31,医药生物,5.964946e+10,2.163841e+09,2.157236e+09,1.401893e+09,5.998204e+09,3.959453e+10,...,-3.020958e+09,1.502747e+09,1,None,None,None,None,None,None,None
1,000028,国药一致,2021-12-31,医药生物,6.835781e+10,1.952411e+09,1.974853e+09,1.336428e+09,5.126159e+09,4.278368e+10,...,-6.716010e+08,1.655181e+09,1,None,None,None,None,None,None,None
2,000028,国药一致,2022-12-31,医药生物,7.344314e+10,2.306069e+09,2.311070e+09,1.486708e+09,6.016350e+09,4.261576e+10,...,7.086612e+08,2.560431e+09,1,None,None,None,None,None,None,None
3,000028,国药一致,2023-12-31,医药生物,7.547748e+10,2.467425e+09,2.473227e+09,1.599256e+09,6.502348e+09,4.757109e+10,...,3.931210e+08,2.937139e+09,1,None,None,None,None,None,None,None
4,000028,国药一致,2024-12-31,医药生物,7.437849e+10,5.734783e+08,5.754694e+08,6.424934e+08,7.408540e+09,4.755586e+10,...,1.288350e+09,3.271468e+09,1,None,None,None,None,None,None,None


---
## Cell 2：下载当前批次年报PDF

**每次运行前修改 `BATCH_START` 和 `BATCH_END`**
- 第1批：0–99
- 第2批：100–199
- 以此类推

In [ ]:
!pip install akshare -q
import requests
import pandas as pd
import akshare as ak
import re
import os
import concurrent.futures
from collections import Counter
# 路径配置
BASE        = '/content/drive/MyDrive/credit_risk_project'
MASTER_PATH = os.path.join(BASE, 'finance_data/master_table.csv')
MASTER2_PATH= os.path.join(BASE, 'master2.csv')
LOG_DIR     = os.path.join(BASE, 'logs')
PDF_DIR     = os.path.join(BASE, 'annual_report_pdf')
# ── 批次设置（每次手动修改这里）────────────────────────
BATCH_START = 300  # ← 改成100继续上次
BATCH_END   = 460
YEAR_START  = '20200601'
YEAR_END    = '20260605'

# ── 读取本批公司列表 ───────────────────────────────────
master2 = pd.read_csv(MASTER2_PATH, dtype={'股票代码': str})
unique_stocks = (
    master2[['股票代码', '股票简称', '申万一级行业', '自然编号']]
    .drop_duplicates(subset='股票代码')
    .sort_values('自然编号')
)
batch = unique_stocks[
    (unique_stocks['自然编号'] >= BATCH_START) &
    (unique_stocks['自然编号'] <= BATCH_END)
].reset_index(drop=True)

print(f'本批公司数：{len(batch)}')
print(batch[['自然编号', '股票代码', '股票简称', '申万一级行业']].to_string())

# ── HTTP Headers ──────────────────────────────────────
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
                  'AppleWebKit/537.36 (KHTML, like Gecko) '
                  'Chrome/120.0.0.0 Safari/537.36',
    'Referer': 'http://www.cninfo.com.cn/',
    'Accept-Language': 'zh-CN,zh;q=0.9',
}

# ── 函数定义（原来的，不动）───────────────────────────
def get_annual_report_urls(symbol):
    try:
        df = ak.stock_zh_a_disclosure_report_cninfo(
            symbol=symbol,
            market='沪深京',
            category='年报',
            start_date=YEAR_START,
            end_date=YEAR_END
        )
        mask = (
            df['公告标题'].str.contains(r'\d{4}年年度报告$', regex=True) &
            ~df['公告标题'].str.contains('摘要|英文|English|H股', na=False)
        )
        return df[mask][['公告标题', '公告时间', '公告链接']].reset_index(drop=True)
    except Exception:
        return None

def build_pdf_url(detail_url, announcement_time):
    ann_id = re.search(r'announcementId=(\d+)', detail_url)
    if ann_id:
        date_str = announcement_time[:10]
        return f"http://static.cninfo.com.cn/finalpage/{date_str}/{ann_id.group(1)}.PDF"
    return None

def download_pdf(pdf_url, save_path):
    try:
        resp = requests.get(pdf_url, headers=HEADERS, timeout=60, stream=True)
        if resp.status_code == 200:
            with open(save_path, 'wb') as f:
                for chunk in resp.iter_content(chunk_size=8192):
                    f.write(chunk)
            return True
        return False
    except Exception:
        return False

# ── 改动1：每家公司打包成一个任务函数 ─────────────────
def process_company(row):
    symbol  = row['股票代码']
    name    = row['股票简称']
    enum_no = row['自然编号']
    results = []

    reports = get_annual_report_urls(symbol)
    if reports is None or len(reports) == 0:
        print(f'⚠️ [{enum_no}] {symbol} {name} 未找到年报')
        return [{'symbol': symbol, 'name': name, 'year': 'ALL',
                 'status': 'no_report', 'pdf_path': ''}]

    for _, rep in reports.iterrows():
        title      = rep['公告标题']
        ann_time   = rep['公告时间']
        link       = rep['公告链接']
        year_match = re.search(r'(\d{4})年', title)
        year       = year_match.group(1) if year_match else 'unknown'
        pdf_url    = build_pdf_url(link, ann_time)
        save_path  = os.path.join(PDF_DIR, f'{symbol}_{year}.pdf')

        if os.path.exists(save_path):
            print(f'  ✓ [{enum_no}] {symbol} {year} 已存在，跳过')
            results.append({'symbol': symbol, 'name': name, 'year': year,
                            'status': 'skipped', 'pdf_path': save_path})
            continue

        if pdf_url is None:
            print(f'  ⚠️ [{enum_no}] {symbol} {year} 无法拼接链接')
            results.append({'symbol': symbol, 'name': name, 'year': year,
                            'status': 'url_failed', 'pdf_path': ''})
            continue

        success = download_pdf(pdf_url, save_path)
        if success:
            size_mb = os.path.getsize(save_path) / 1024 / 1024
            print(f'  ✅ [{enum_no}] {symbol} {year} ({size_mb:.1f}MB)')
            results.append({'symbol': symbol, 'name': name, 'year': year,
                            'status': 'success', 'pdf_path': save_path})
        else:
            print(f'  ❌ [{enum_no}] {symbol} {year} 下载失败')
            results.append({'symbol': symbol, 'name': name, 'year': year,
                            'status': 'download_failed', 'pdf_path': ''})
    return results

# ── 改动2：并发执行，去掉所有sleep ────────────────────
download_results = []
rows = [row for _, row in batch.iterrows()]

with concurrent.futures.ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(process_company, row) for row in rows]
    for future in concurrent.futures.as_completed(futures):
        download_results.extend(future.result())

status_count = Counter(r['status'] for r in download_results)
print(f'\n本批下载完成，共{len(download_results)}条记录')
print(f'成功:{status_count["success"]} 跳过:{status_count["skipped"]} 失败:{status_count["download_failed"]}')

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 100.0 MB/s eta 0:00:00
本批公司数：159
     自然编号    股票代码  股票简称 申万一级行业
0     300  603014  威高血净   医药生物
1     301  603087  甘李药业   医药生物
2     302  603108  润达医疗   医药生物
3     303  603122  合富中国   医药生物
4     304  603127  昭衍新药   医药生物
5     305  603139  康惠股份   医药生物
6     306  603168  莎普爱思   医药生物
7     307  603205   健尔康   医药生物
8     308  603207  小方制药   医药生物
9     309  603222  济民健康   医药生物
10    310  603229  奥翔药业   医药生物
11    311  603233   大参林   医药生物
12    312  603259  药明康德   医药生物
13    313  603301  振德医疗   医药生物
14    314  603309  维力医疗   医药生物
15    315  603351  威尔药业   医药生物
16    316  603367  辰欣药业   医药生物
17    317  603368  柳药集团   医药生物
18    318  603387  基蛋生物   医药生物
19    319  603392  万泰生物   医药生物
20    320  603439  三力制药   医药生物
21    321  603456  九洲药业   医药生物
22

  0%|          | 0/1 [00:00<?, ?it/s]



  ✅ [304] 603127 2025 (1.4MB)
  ✅ [303] 603122 2025 (1.3MB)




100%|██████████| 1/1 [00:00<00:00,  1.57it/s]

                                             

  ✅ [301] 603087 2025 (1.5MB)
  ✅ [303] 603122 2024 (3.9MB)
  ❌ [304] 603127 2024 下载失败
  ✅ [300] 603014 2025 (2.1MB)
  ✅ [301] 603087 2024 (1.8MB)
  ✅ [303] 603122 2023 (3.8MB)
  ✓ [303] 603122 2023 已存在，跳过
  ✅ [304] 603127 2023 (3.8MB)
  ✅ [301] 603087 2023 (3.9MB)


...


100%|██████████| 1/1 [00:00<00:00,  1.54it/s]
                                             

  ✅ [455] 688767 2024 (1.5MB)
  ✅ [457] 688799 2025 (1.7MB)
  ✅ [450] 688739 2022 (4.1MB)


  0%|          | 0/1 [00:00<?, ?it/s]

  ✅ [456] 688796 2025 (1.8MB)
  ✅ [457] 688799 2024 (1.6MB)
  ✅ [455] 688767 2023 (4.3MB)
  ✅ [450] 688739 2021 (4.3MB)


  ✅ [455] 688767 2022 (3.8MB)
  ✅ [457] 688799 2023 (4.5MB)
  ✅ [458] 688805 2025 (1.5MB)
  ✅ [457] 688799 2022 (4.2MB)
  ✅ [457] 688799 2021 (4.1MB)

本批下载完成，共791条记录
成功:775 跳过:14 失败:1


---
## Cell 3：写下载日志

In [ ]:
from datetime import datetime

# 生成日志文件名（含批次和时间）
log_filename = f'batch_{BATCH_START:04d}_{BATCH_END:04d}_{datetime.now().strftime("%Y%m%d_%H%M")}.csv'
log_path = os.path.join(LOG_DIR, log_filename)

# 转为DataFrame写入
log_df = pd.DataFrame(download_results)
log_df['batch']      = f'{BATCH_START}-{BATCH_END}'
log_df['log_time']   = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
log_df.to_csv(log_path, index=False, encoding='utf-8-sig')

# 打印汇总
print(f'日志已保存：{log_path}')
print(f'\n===== 本批下载汇总 =====')
print(log_df['status'].value_counts())
print(f'\n失败列表：')
failed = log_df[log_df['status'].isin(['download_failed', 'url_failed', 'no_report'])]
if len(failed) > 0:
    print(failed[['symbol', 'name', 'year', 'status']].to_string())
else:
    print('无失败记录 ✅')

日志已保存：/content/drive/MyDrive/credit_risk_project/logs/batch_0300_0460_20260612_0144.csv

===== 本批下载汇总 =====
status
success            775
skipped             14
download_failed      1
no_report            1
Name: count, dtype: int64

失败列表：
    symbol  name  year           status
9   603127  昭衍新药  2024  download_failed
51  603108  润达医疗   ALL        no_report


In [ ]:

df = pd.read_csv('/content/drive/MyDrive/credit_risk_project/master2.csv', dtype={'股票代码': str})
df = df[df['股票代码'] != '603108']
df.to_csv('/content/drive/MyDrive/credit_risk_project/master2.csv', index=False, encoding='utf-8-sig')
print(f'剩余记录：{len(df)}条')


剩余记录：2740条


---
## Cell 4：提取MDA文本，保存为CSV

In [ ]:
# ── Cell 1：函数定义（只跑一次）────────────────────────
!pip install pdfplumber transformers torch sentencepiece -q
import pdfplumber
import pandas as pd
import os
import re
import logging
import shutil
from datetime import datetime

MDA_START_KEYWORDS = ['管理层讨论与分析', '经营情况讨论与分析', '管理层的讨论与分析']
MDA_END_KEYWORDS   = ['监事会工作报告', '监事会报告', '重要事项', '重大事项',
                      '公司治理', '股本变动', '财务报告', '独立审计师', '审计报告']

def is_toc_page(text):
    return len(re.findall(r'[．。·\.]{3,}', text)) >= 3

def is_section_title(line):
    line = line.strip()
    if not line or len(line) > 20:
        return False
    for kw in MDA_END_KEYWORDS:
        if kw in line:
            idx = line.find(kw) + len(kw)
            if idx < len(line) and line[idx] in '"。分一中关之》"—"第的':
                return False
    return True

def extract_text_no_tables(page):
    tables = page.find_tables()
    if not tables:
        return page.extract_text() or ''
    table_bboxes = [t.bbox for t in tables]
    def not_in_table(obj):
        for x0, top, x1, bottom in table_bboxes:
            if obj['x0'] >= x0 and obj['x1'] <= x1 and obj['top'] >= top and obj['bottom'] <= bottom:
                return False
        return True
    return page.filter(not_in_table).extract_text() or ''

def extract_mda(pdf_path):
    try:
        with pdfplumber.open(pdf_path) as pdf:
            total_pages = len(pdf.pages)
            start_page = end_page = None

            for i, page in enumerate(pdf.pages):
                text = page.extract_text() or ''  # 扫描阶段用普通版，快
                if not text.strip():
                    continue
                lines = [l.strip() for l in text.splitlines() if l.strip()]

                if start_page is None:
                    for line in lines:
                        if any(kw in line for kw in MDA_START_KEYWORDS) and not is_toc_page(text):
                            start_page = i
                            break
                else:
                    for line in lines[:5]:
                        if any(kw in line for kw in MDA_END_KEYWORDS) and not is_toc_page(text) and is_section_title(line):
                            end_page = i
                            break

                if end_page is not None:
                    break

            if start_page is None:
                return None, 'toc_not_found'

            p_end = end_page if end_page else min(total_pages, start_page + 100)
            mda_text = ''
            for i in range(start_page, p_end):
                t = extract_text_no_tables(pdf.pages[i])  # 提取正文时去表格
                if t:
                    mda_text += t + '\n'

            if len(mda_text.strip()) < 500:
                return None, 'scanned_pdf'

            return mda_text.strip(), 'success'

    except Exception as e:
        return None, f'error:{str(e)[:50]}'

def process_one(rec):
    symbol, year, path = rec['symbol'], rec['year'], rec['pdf_path']
    mda_text, status = extract_mda(path)
    if status == 'success':
        logger.info(f'✅ {symbol} {year} {len(mda_text)}字')
    else:
        logger.warning(f'❌ {symbol} {year} {status}')
    return {'股票代码': symbol, '年份': year, 'mda_text': mda_text or '', 'extract_status': status}

print('函数定义完成')

函数定义完成


In [ ]:
# 路径配置 + 执行
BASE        = '/content/drive/MyDrive/credit_risk_project'
PDF_DIR     = os.path.join(BASE, 'annual_report_pdf')
OUT_CSV     = os.path.join(BASE, 'MDA_txt.csv')
LOG_DIR     = os.path.join(BASE, 'logs')
os.makedirs(LOG_DIR, exist_ok=True)

# 日志
for h in logging.root.handlers[:]:
    logging.root.removeHandler(h)
log_path = os.path.join(LOG_DIR, f'mda_extract_{datetime.now().strftime("%Y%m%d_%H%M%S")}.log')
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(message)s',
    handlers=[
        logging.FileHandler(log_path, encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# 加载已有结果
if os.path.exists(OUT_CSV):
    existing_df = pd.read_csv(OUT_CSV, dtype={'股票代码': str, '年份': str})
    done_keys = set(zip(
        existing_df[existing_df['extract_status'] == 'success']['股票代码'],
        existing_df[existing_df['extract_status'] == 'success']['年份'].astype(str)
    ))
    logger.info(f'已有记录：{len(existing_df)}条')
else:
    existing_df = pd.DataFrame()
    done_keys   = set()

# 扫描PDF文件夹
all_pdfs = []
for fname in sorted(os.listdir(PDF_DIR)):
    if not fname.endswith('.pdf'):
        continue
    parts = fname.replace('.pdf', '').split('_')
    if len(parts) == 2:
        all_pdfs.append({'symbol': parts[0], 'year': parts[1], 'pdf_path': os.path.join(PDF_DIR, fname)})

pending = [r for r in all_pdfs if (r['symbol'], r['year']) not in done_keys]
logger.info(f'共{len(all_pdfs)}个PDF，已处理{len(done_keys)}个，待处理{len(pending)}个')

# 分批复制+提取
new_rows = []
total = len(pending)
LOCAL_PDF_DIR = '/tmp/annual_report_pdf'
os.makedirs(LOCAL_PDF_DIR, exist_ok=True)
BATCH_SIZE = 50

for batch_start in range(0, len(pending), BATCH_SIZE):
    batch = pending[batch_start:batch_start + BATCH_SIZE]

    print(f'复制第{batch_start+1}-{batch_start+len(batch)}个PDF...')
    for rec in batch:
        shutil.copy2(rec['pdf_path'], os.path.join(LOCAL_PDF_DIR, os.path.basename(rec['pdf_path'])))

    for i, rec in enumerate(batch, batch_start + 1):
        local_rec = {**rec, 'pdf_path': os.path.join(LOCAL_PDF_DIR, os.path.basename(rec['pdf_path']))}
        try:
            result = process_one(local_rec)
            new_rows.append(result)
            status = result['extract_status']
            length = len(result['mda_text'])
            print(f'[{i}/{total}] {"✅" if status=="success" else "❌"} {rec["symbol"]} {rec["year"]} {status} {length}字')
        except Exception as e:
            logger.error(f'💥 {rec["symbol"]} {rec["year"]} 异常：{e}')
            new_rows.append({'股票代码': rec['symbol'], '年份': rec['year'], 'mda_text': '', 'extract_status': f'error:{str(e)[:50]}'})
            print(f'[{i}/{total}] 💥 {rec["symbol"]} {rec["year"]} 异常')

    for rec in batch:
        local_path = os.path.join(LOCAL_PDF_DIR, os.path.basename(rec['pdf_path']))
        if os.path.exists(local_path):
            os.remove(local_path)

    if new_rows:
        temp_df = pd.DataFrame(new_rows)
        combined = pd.concat([existing_df, temp_df], ignore_index=True)
        combined.to_csv(OUT_CSV, index=False, encoding='utf-8-sig')
        logger.info(f'中间保存：{i}/{total}')

# 最终保存
if new_rows:
    new_df   = pd.DataFrame(new_rows)
    combined = pd.concat([existing_df, new_df], ignore_index=True)
    combined.to_csv(OUT_CSV, index=False, encoding='utf-8-sig')
    success  = new_df[new_df['extract_status'] == 'success']
    fail     = new_df[new_df['extract_status'] != 'success']
    logger.info(f'完成：新增{len(new_rows)}条（成功{len(success)}，失败{len(fail)}），累计{len(combined)}条')
    logger.info(f'CSV已保存：{OUT_CSV}')
    logger.info(f'日志已保存：{log_path}')
else:
    logger.info('本批次无新增记录')

2026-06-12 01:54:34,722 INFO 已有记录：1814条
2026-06-12 01:54:34,862 INFO 共775个PDF，已处理1651个，待处理775个


复制第1-50个PDF...


2026-06-12 01:54:54,709 INFO ✅ 603014 2025 14008字


[1/775] ✅ 603014 2025 success 14008字


2026-06-12 01:55:00,516 INFO ✅ 603087 2020 11943字


...

2026-06-12 04:09:15,865 INFO ✅ 688799 2025 45244字


[774/775] ✅ 688799 2025 success 45244字


2026-06-12 04:09:24,274 INFO ✅ 688805 2025 25965字


[775/775] ✅ 688805 2025 success 25965字


2026-06-12 04:09:27,327 INFO 中间保存：775/775
2026-06-12 04:09:30,487 INFO 完成：新增775条（成功773，失败2），累计2589条
2026-06-12 04:09:30,489 INFO CSV已保存：/content/drive/MyDrive/credit_risk_project/MDA_txt.csv
2026-06-12 04:09:30,493 INFO 日志已保存：/content/drive/MyDrive/credit_risk_project/logs/mda_extract_20260612_015430.log


In [ ]:
import pandas as pd

OUT_CSV = '/content/drive/MyDrive/credit_risk_project/MDA_txt.csv'
df = pd.read_csv(OUT_CSV, dtype={'股票代码': str})
fail_df = df[df['extract_status'] != 'success']
print(fail_df[['股票代码', '年份', 'extract_status']].sort_values('股票代码'))

        股票代码    年份 extract_status
87    000597  2023  toc_not_found
100   000597  2023  toc_not_found
112   000705  2019  toc_not_found
113   000705  2020  toc_not_found
115   000705  2022  toc_not_found
...      ...   ...            ...
1718  600976  2020  toc_not_found
1721  600976  2024  toc_not_found
1723  600993  2020  toc_not_found
1844  603205  2025  toc_not_found
2131  688108  2023    scanned_pdf

[165 rows x 3 columns]


---
## Cell 5：FinBERT情感分析，写回master2



In [ ]:
test = nlp(['公司面临严重亏损风险'], top_k=3)
print(test)

[[{'label': 'Negative', 'score': 0.9987032413482666}, {'label': 'Positive', 'score': 0.0009250383009202778}, {'label': 'Neutral', 'score': 0.0003716982319019735}]]


In [ ]:
from transformers import pipeline
import pandas as pd
import os
import torch

# 路径
BASE         = '/content/drive/MyDrive/credit_risk_project'
MDA_CSV_PATH = os.path.join(BASE, 'MDA_txt.csv')  # ← 路径修正
MASTER2_PATH = os.path.join(BASE, 'master2.csv')

# 风险/不确定词典
RISK_WORDS = [
    '风险', '亏损', '诉讼', '违约', '下滑', '减值', '不确定', '困难',
    '压力', '损失', '偿还', '逾期', '违规', '处罚', '调查',
    '退市', '警示', '违法', '欺诈', '造假', '负债累累'
]
UNCERTAINTY_WORDS = [
    '尚难以预计', '存在不确定', '无法确定', '难以判断',
    '存在一定影响', '或将', '预计将', '可能存在'
]

# 加载FinBERT
device = 0 if torch.cuda.is_available() else -1
print(f'使用设备：{"GPU" if device == 0 else "CPU"}')

nlp = pipeline(
    'text-classification',
    model='yiyanghkust/finbert-tone-chinese',
    tokenizer='yiyanghkust/finbert-tone-chinese',
    device=device
)
print('模型加载完成 ✅')

# 切片与打分函数
def split_text(text, max_len=500):
    chunks = []
    for i in range(0, len(text), max_len):
        chunk = text[i:i+max_len].strip()
        if len(chunk) > 50:
            chunks.append(chunk)
    return chunks

def analyze_mda(text):
    if not text or len(text.strip()) < 100:
        return None
    chunks = split_text(text, max_len=500)
    if not chunks:
        return None


    results = nlp(chunks, truncation=True, max_length=512, top_k=3)
    lengths = [len(c) for c in chunks]
    total_len = sum(lengths)
    weights = [l / total_len for l in lengths]

    neg_scores = []
    for r in results:
        neg_prob = next((x['score'] for x in r if x['label'] == 'Negative'), 0.0)
        neg_scores.append(neg_prob)


    negative_score = sum(s * w for s, w in zip(neg_scores, weights))
    negative_score_max = max(neg_scores)
    risk_count        = sum(text.count(w) for w in RISK_WORDS)
    uncertainty_count = sum(text.count(w) for w in UNCERTAINTY_WORDS)

    return {
        'negative_score'    : round(negative_score, 4),
        'negative_score_max' : round(negative_score_max, 4),
        'risk_word_ratio'   : round(risk_count / total_len * 1000, 4),
        'uncertainty_ratio' : round(uncertainty_count / total_len * 1000, 4),
        'mda_length'        : total_len
    }

# 读取数据
mda_df  = pd.read_csv(MDA_CSV_PATH, dtype={'股票代码': str})
master2 = pd.read_csv(MASTER2_PATH, dtype={'股票代码': str})
master2['报告期年份'] = pd.to_datetime(master2['报告期']).dt.year.astype(str)

# 所有提取成功的记录
to_process = mda_df[
    (mda_df['extract_status'] == 'success') &
    (mda_df['mda_text'].notna()) &
    (mda_df['mda_text'] != '')
].reset_index(drop=True)


done_keys = set(
    zip(
        master2[master2['nlp_status'] == 'success']['股票代码'],
        master2[master2['nlp_status'] == 'success']['报告期年份'].astype(str)
    )
)
to_process = to_process[
    ~to_process.apply(lambda r: (r['股票代码'], str(r['年份'])) in done_keys, axis=1)
].reset_index(drop=True)


print(f'待分析：{len(to_process)}篇（已跳过{len(done_keys)}篇）')



# 确保master2有NLP列
for col in ['negative_score', 'negative_score_max','risk_word_ratio', 'uncertainty_ratio', 'mda_length', 'nlp_status']:
    if col not in master2.columns:
        master2[col] = None

# 主循环
success_count = fail_count = 0

for i, row in to_process.iterrows():
    symbol = row['股票代码']
    year   = str(row['年份'])
    text   = row['mda_text']

    print(f'[{i+1}/{len(to_process)}] {symbol} {year}...', end=' ')

    try:
        result = analyze_mda(text)
        if result:
            mask = (master2['股票代码'] == symbol) & (master2['报告期年份'] == year)
            for col in ['negative_score','negative_score_max', 'risk_word_ratio', 'uncertainty_ratio', 'mda_length']:
                master2.loc[mask, col] = result[col]
            master2.loc[mask, 'nlp_status'] = 'success'
            print(f'✅ neg={result["negative_score"]:.3f} risk={result["risk_word_ratio"]:.2f}')
            success_count += 1
        else:
            master2.loc[
                (master2['股票代码'] == symbol) & (master2['报告期年份'] == year),
                'nlp_status'
            ] = 'empty_text'
            print('❌ 文本为空')
            fail_count += 1
    except Exception as e:
        print(f'❌ 错误: {str(e)[:50]}')
        fail_count += 1

    # 每50篇保存一次
    if (i + 1) % 50 == 0:
        master2.drop(columns=['报告期年份'], errors='ignore').to_csv(MASTER2_PATH, index=False, encoding='utf-8-sig')
        print(f'--- 中间保存：{i+1}/{len(to_process)} ---')

# 最终保存
master2.drop(columns=['报告期年份'], inplace=True)
master2.to_csv(MASTER2_PATH, index=False, encoding='utf-8-sig')

print(f'\nNLP分析完成')
print(f'成功：{success_count} 篇，失败：{fail_count} 篇')
print(f'master2已更新：{MASTER2_PATH}')

# 预览
print('\nNLP结果预览：')
preview = master2[master2['nlp_status'] == 'success'][
    ['股票代码', '报告期', 'negative_score', 'risk_word_ratio', 'uncertainty_ratio', 'mda_length']
]
print(preview.head(10).to_string())

使用设备：GPU


模型加载完成 ✅
待分析：861篇（已跳过1563篇）
[1/861] 000028 2019... ✅ neg=0.100 risk=1.76
[2/861] 000153 2019... ✅ neg=0.091 risk=1.68
[3/861] 000403 2019... ✅ neg=0.052 risk=2.31
[4/861] 000411 2019... ✅ neg=0.009 risk=3.22
[5/861] 000423 2019... ✅ neg=0.066 risk=1.00
[6/861] 000513 2019... ✅ neg=0.074 risk=1.20
[7/861] 000516 2019... ✅ neg=0.036 risk=0.71
[8/861] 000518 2019... ✅ neg=0.002 risk=0.68
[9/861] 000534 2019... ✅ neg=0.119 risk=1.24
[10/861] 000538 2019... ✅ neg=0.035 risk=1.02
[11/861] 000566 2019... ✅ neg=0.102 risk=1.22
[12/861] 000590 2019... ✅ neg=0.061 risk=1.88
[13/861] 000597 2019... ✅ neg=0.050 risk=1.83
[14/861] 000623 2019... ✅ neg=0.086 risk=0.62
[15/861] 000650 2019... ✅ neg=0.002 risk=0.39
[16/861] 000661 2019... ✅ neg=0.096 risk=0.60
[17/861] 000710 2019... ✅ neg=0.098 risk=1.81
[18/861] 000739 2019... ✅ neg=0.001 risk=0.29
[19/861] 000790 2019... ✅ neg=0.019 risk=1.55
[20/861] 000813 2019... ✅ neg=0.138 risk=2.35
[21/861] 000919 2019... ✅ neg=0.182 risk=3.52
[22/861] 000931

In [ ]:
import shutil, os
PDF_DIR = '/content/drive/MyDrive/credit_risk_project/annual_report_pdf'
shutil.rmtree(PDF_DIR)
os.makedirs(PDF_DIR)
print('已清空')

已清空


In [ ]:
import pandas as pd

finance = pd.read_csv('/content/drive/MyDrive/credit_risk_project/finance_data/master_table.csv', dtype={'股票代码': str})
master2 = pd.read_csv('/content/drive/MyDrive/credit_risk_project/master2.csv', dtype={'股票代码': str})

print('finance table')
print(finance.shape)
print(finance.head())

print('\n master2')
print(master2.shape)
print(master2.head())

finance table
(30389, 15)
     股票代码 股票简称         报告期 申万一级行业         营业总收入          营业利润          利润总额  \
0  000002  万科A  2020-12-31    房地产  4.191117e+11  7.995864e+10  7.967575e+10   
1  000002  万科A  2021-12-31    房地产  4.527978e+11  5.253100e+10  5.222263e+10   
2  000002  万科A  2022-12-31    房地产  5.038384e+11  5.202905e+10  5.240830e+10   
3  000002  万科A  2023-12-31    房地产  4.657391e+11  2.925170e+10  2.980543e+10   
4  000002  万科A  2024-12-31    房地产  3.431764e+11 -4.564380e+10 -4.718656e+10   

            净利润       资产-货币资金        资产-总资产        负债-总负债      资产负债率  \
0  4.151554e+10  1.952307e+11  1.869177e+12  1.519333e+12  81.283503   
1  2.252403e+10  1.493524e+11  1.938638e+12  1.545865e+12  79.739758   
2  2.268855e+10  1.372076e+11  1.757805e+12  1.352168e+12  76.923672   
3  1.216268e+10  9.981376e+10  1.504850e+12  1.101917e+12  73.224342   
4 -4.947843e+10  8.816287e+10  1.286260e+12  9.474052e+11  73.655816   

         股东权益合计     净现金流-净现金流  经营性现金流-现金流量净额  
0  3.498445e+11  2.

In [ ]:
# 缺失值检查
print('缺失值统计 ')
missing = master2.isnull().sum()
missing_pct = (missing / len(master2) * 100).round(2)
missing_df = pd.DataFrame({'缺失数': missing, '缺失率%': missing_pct})
print(missing_df[missing_df['缺失数'] > 0].sort_values('缺失率%', ascending=False))

print(f'\n总行数：{len(master2)}')
print(f'\n所有列名：')
print(master2.columns.tolist())

缺失值统计 
                     缺失数    缺失率%
sentiment_score     2740  100.00
positive_ratio      2740  100.00
negative_ratio      2740  100.00
mda_length           404   14.74
risk_word_ratio      404   14.74
uncertainty_ratio    404   14.74
negative_score       404   14.74
nlp_status           404   14.74
negative_score_max   404   14.74
营业总收入                 12    0.44

总行数：2740

所有列名：
['股票代码', '股票简称', '报告期', '申万一级行业', '营业总收入', '营业利润', '利润总额', '净利润', '资产-货币资金', '资产-总资产', '负债-总负债', '资产负债率', '股东权益合计', '净现金流-净现金流', '经营性现金流-现金流量净额', '自然编号', 'sentiment_score', 'positive_ratio', 'negative_ratio', 'risk_word_ratio', 'uncertainty_ratio', 'mda_length', 'nlp_status', 'negative_score', 'negative_score_max']


In [ ]:
master2.drop(columns=['sentiment_score', 'positive_ratio', 'negative_ratio'], inplace=True)
master2.to_csv('/content/drive/MyDrive/credit_risk_project/master2.csv', index=False, encoding='utf-8-sig')
print(f'剩余列数：{len(master2.columns)}')
print(master2.columns.tolist())

剩余列数：22
['股票代码', '股票简称', '报告期', '申万一级行业', '营业总收入', '营业利润', '利润总额', '净利润', '资产-货币资金', '资产-总资产', '负债-总负债', '资产负债率', '股东权益合计', '净现金流-净现金流', '经营性现金流-现金流量净额', '自然编号', 'risk_word_ratio', 'uncertainty_ratio', 'mda_length', 'nlp_status', 'negative_score', 'negative_score_max']


In [ ]:
# 查看404条缺失的nlp_status分布
print(master2[master2['negative_score'].isna()]['nlp_status'].value_counts(dropna=False))

# 营业总收入12条缺失直接删行（0.44%，影响不大）
master2 = master2[master2['营业总收入'].notna()]

print(f'清理后行数：{len(master2)}')

nlp_status
NaN    404
Name: count, dtype: int64
清理后行数：2728


In [ ]:
master2 = master2[master2['negative_score'].notna()].reset_index(drop=True)
master2.to_csv('/content/drive/MyDrive/credit_risk_project/master2.csv', index=False, encoding='utf-8-sig')
print(f'剩余行数：{len(master2)}')

剩余行数：2333
